# HGSR-GSX with PPO (Reinforcement Learning)

## Metrics:
- D (Diversity) = stable_rank = Σσ²/σ_max²
- S (Scale) = trace = Σσ
- Episodic Return, Dormant Neurons, Weight Magnitude

In [ ]:
# =================================================================
# STEP 1: Install dependencies (RUN THIS CELL FIRST, then RESTART)
# =================================================================
!pip install gymnasium[mujoco] mujoco mlproj-manager scipy matplotlib tqdm

In [ ]:
import os, sys, time, pickle
from copy import deepcopy
import numpy as np
import torch
import torch.nn as nn
from tqdm import tqdm
import matplotlib.pyplot as plt

# Use gymnasium (modern, compatible with Python 3.12)
import gymnasium as gym
print(f"gymnasium version: {gym.__version__}")
sys.path.append("/kaggle/input/lop-src/lop")
from lop.nets.policies import MLPPolicy
from lop.nets.valuefs import MLPVF
from lop.algos.rl.buffer import Buffer
from lop.utils.miscellaneous import compute_matrix_rank_summaries

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## HGSR-GSX for RL (Actor-Critic)

In [ ]:
class HGSR_GSX_RL:
    """
    HGSR-GSX algorithm adapted for Reinforcement Learning (Actor-Critic).
    Monitors both policy and value networks for diversity (D) and scale (S).
    """
    DEFAULT_K = 20
    DEFAULT_T_CHECK = 1000  # Check every N steps
    DEFAULT_TAU_D = 0.4
    DEFAULT_EPSILON = 0.25
    DEFAULT_ALPHA_0 = 0.1
    
    def __init__(self, pol, vf, lr=0.0001, g=0.99, lm=0.95, device='cuda',
                 K=None, T_check=None, tau_D=None, epsilon=0.15, alpha_0=None,
                 grad_clip=1.0, h_dim=(256, 256), bs=2048, n_itrs=10, n_slices=16,
                 clip_eps=0.2, wd=0, betas=(0.9, 0.999), eps=1e-8):
        
        self.K = K or self.DEFAULT_K
        self.T_check = T_check or self.DEFAULT_T_CHECK
        self.tau_D = tau_D or self.DEFAULT_TAU_D
        self.epsilon = epsilon or self.DEFAULT_EPSILON
        self.alpha_0 = alpha_0 or self.DEFAULT_ALPHA_0
        self.grad_clip = grad_clip
        
        self.pol = pol
        self.vf = vf
        self.g = g  # discount
        self.lm = lm  # lambda for GAE
        self.bs = bs
        self.n_itrs = n_itrs
        self.n_slices = n_slices
        self.clip_eps = clip_eps
        
        self.device = device
        self.lr = lr
        
        # Combined optimizer for policy and value function
        self.opt = torch.optim.Adam(
            list(pol.parameters()) + list(vf.parameters()),
            lr=lr, weight_decay=wd, betas=betas, eps=eps
        )
        
        self.step = 0
        self.grad_buffers_pol = {}
        self.grad_buffers_vf = {}
        self.initial_D = {}
        self.initial_S = {}
        
        self.hard_reset_count = 0
        self.expand_count = 0
        
        # TRACKING
        self.spectrum_history = []
        self.eigenvalue_history = []
        self.reset_log = []
        self.expand_log = []
        
        # Initialize grad buffers for policy network
        for name, p in self.pol.named_parameters():
            if 'weight' in name and p.dim() >= 2:
                self.grad_buffers_pol[name] = []
                
        # Initialize grad buffers for value network
        for name, p in self.vf.named_parameters():
            if 'weight' in name and p.dim() >= 2:
                self.grad_buffers_vf[name] = []
        
        print(f"HGSR-GSX-RL: tau_D={self.tau_D}, epsilon={self.epsilon}, T_check={self.T_check}")
    
    def _compute_metrics(self, w, return_eigenvalues=False):
        """Compute D, S, and optionally top-k eigenvalues"""
        w = w.view(w.size(0), -1)
        try:
            s = torch.linalg.svdvals(w)
            D = ((s ** 2).sum() / (s[0] ** 2 + 1e-8)).item()
            S = s.sum().item()
            if return_eigenvalues:
                top_k = min(10, len(s))
                return D, S, s[:top_k].cpu().numpy()
            return D, S
        except:
            if return_eigenvalues:
                return 1.0, 1.0, np.array([1.0])
            return 1.0, 1.0
    
    def _store_gradient(self, name, grad, buffers):
        if name not in buffers: return
        buf = buffers[name]
        buf.append(grad.view(-1).detach().clone())
        if len(buf) > self.K: buf.pop(0)
    
    def _build_subspace(self, name, buffers):
        buf = buffers[name]
        if len(buf) < 3: return None
        try:
            G = torch.stack(buf, dim=1)
            U, S, _ = torch.linalg.svd(G, full_matrices=False)
            r = min(len(buf), 10, max(1, (S > S[0] * 0.1).sum().item()))
            return U[:, :r]
        except:
            return None
    
    def _orthogonal_noise(self, U_r, d, device):
        z = torch.randn(d, device=device)
        if U_r is not None:
            z = z - U_r @ (U_r.T @ z)
        return z / (z.norm() + 1e-8)
    
    def _hard_reset_param(self, param, name):
        """Reinitialize a single parameter"""
        with torch.no_grad():
            if param.dim() == 2:
                nn.init.kaiming_uniform_(param, nonlinearity='relu')
            elif param.dim() == 1:
                param.zero_()
        self.hard_reset_count += 1
    
    def _check_network(self, net, grad_buffers, net_name):
        """Check and intervene on a single network"""
        hard_resets, expansions = 0, 0
        
        for name, param in net.named_parameters():
            if name not in grad_buffers: continue
            
            D, S, eigvals = self._compute_metrics(param.data, return_eigenvalues=True)
            
            full_name = f"{net_name}.{name}"
            if full_name not in self.initial_D:
                self.initial_D[full_name] = D
                self.initial_S[full_name] = S
                continue
            
            D_ratio = D / (self.initial_D[full_name] + 1e-8)
            S_ratio = S / (self.initial_S[full_name] + 1e-8)
            
            self.spectrum_history.append({
                'step': self.step, 'layer': full_name,
                'D': D, 'S': S, 'D_ratio': D_ratio, 'S_ratio': S_ratio
            })
            
            self.eigenvalue_history.append({
                'step': self.step, 'layer': full_name, 'eigenvalues': eigvals
            })
            
            # Decision
            if S_ratio <= self.epsilon:
                self._hard_reset_param(param, name)
                grad_buffers[name] = []
                self.initial_D[full_name] = D
                self.initial_S[full_name] = S
                hard_resets += 1
                self.reset_log.append({'step': self.step, 'layer': full_name, 'S_ratio': S_ratio})
            elif D_ratio < self.tau_D:
                U_r = self._build_subspace(name, grad_buffers)
                xi = self._orthogonal_noise(U_r, param.numel(), param.device)
                alpha = self.alpha_0 * (1 - D_ratio / self.tau_D)
                if param.grad is not None:
                    param.grad.add_(alpha * xi.view_as(param.grad))
                expansions += 1
                self.expand_count += 1
                self.expand_log.append({'step': self.step, 'layer': full_name, 'D_ratio': D_ratio})
        
        return hard_resets, expansions
    
    def check_and_intervene(self):
        """Check both policy and value networks"""
        if self.step % self.T_check != 0:
            return {'action': 'none', 'hard_resets': 0, 'expansions': 0}
        
        hr_pol, ex_pol = self._check_network(self.pol, self.grad_buffers_pol, 'pol')
        hr_vf, ex_vf = self._check_network(self.vf, self.grad_buffers_vf, 'vf')
        
        hard_resets = hr_pol + hr_vf
        expansions = ex_pol + ex_vf
        action = 'hard_reset' if hard_resets else ('expand' if expansions else 'none')
        return {'action': action, 'hard_resets': hard_resets, 'expansions': expansions}
    
    def get_rets_advs(self, rs, dones, vals):
        """Compute returns and advantages using GAE"""
        dones, rs, vals = dones.to(self.device), rs.to(self.device), vals.to(self.device)
        advs = torch.zeros(len(rs) + 1, device=self.device)
        for t in reversed(range(len(rs))):
            delta = rs[t] + (1 - dones[t]) * self.g * vals[t + 1] - vals[t]
            advs[t] = delta + (1 - dones[t]) * self.g * self.lm * advs[t + 1]
        v_rets = advs[:-1] + vals[:-1]
        advs = advs[:-1].view(-1, 1)
        advs = advs - advs.mean()
        if advs.std() != 0 and not torch.isnan(advs.std()):
            advs = advs / advs.std()
        return v_rets.view(-1, 1).detach(), advs.detach()
    
    def learn(self, buf):
        """PPO learning step"""
        os, acts, rs, op, logpbs, _, dones = buf.get(self.pol.dist_stack)
        
        with torch.no_grad():
            pre_vals = self.vf.value(torch.cat((os, op)))
        v_rets, advs = self.get_rets_advs(rs.squeeze(), dones.squeeze(), pre_vals.squeeze())
        
        inds = np.arange(os.shape[0])
        mini_bs = self.bs // self.n_slices
        
        for _ in range(self.n_itrs):
            np.random.shuffle(inds)
            for start in range(0, len(os), mini_bs):
                ind = inds[start:start + mini_bs]
                
                # Policy loss
                logpts, _ = self.pol.logp_dist(os[ind], acts[ind], to_log_features=True)
                grad_sub = (logpts - logpbs[ind]).exp()
                p_loss0 = -(grad_sub * advs[ind])
                ext_loss = -(torch.clamp(grad_sub, 1 - self.clip_eps, 1 + self.clip_eps) * advs[ind])
                p_loss = torch.max(p_loss0, ext_loss).mean()
                
                # Value loss
                vals = self.vf.value(os[ind], to_log_features=True)
                v_loss = (v_rets[ind] - vals).pow(2).mean()
                
                # Combined loss
                loss = p_loss + v_loss
                
                self.opt.zero_grad()
                loss.backward()
                
                # Store gradients for HGSR-GSX
                for name, p in self.pol.named_parameters():
                    if name in self.grad_buffers_pol and p.grad is not None:
                        self._store_gradient(name, p.grad, self.grad_buffers_pol)
                for name, p in self.vf.named_parameters():
                    if name in self.grad_buffers_vf and p.grad is not None:
                        self._store_gradient(name, p.grad, self.grad_buffers_vf)
                
                if self.grad_clip > 0:
                    nn.utils.clip_grad_norm_(
                        list(self.pol.parameters()) + list(self.vf.parameters()),
                        self.grad_clip
                    )
                self.opt.step()
        
        return {'p_loss': p_loss.item(), 'v_loss': v_loss.item()}
    
    def compute_weight_magnitude(self):
        """Compute average weight magnitude for both networks"""
        total, n = 0.0, 0
        with torch.no_grad():
            for net in [self.pol, self.vf]:
                for name, p in net.named_parameters():
                    if 'weight' in name:
                        total += p.abs().mean().item()
                        n += 1
        return total / n if n else 0.0

In [ ]:
@torch.no_grad()
def compute_dormant_units_rl(pol, vf, threshold=0.01):
    """
    Compute fraction of dormant neurons based on stored activations
    """
    total_units = 0
    dormant_units = 0
    
    # Check policy activations
    if hasattr(pol, 'activations') and pol.activations:
        for key, feat in pol.activations.items():
            if feat is not None and feat.dim() >= 2:
                activity = (feat != 0).float().mean(dim=0)
                dormant = (activity < threshold).sum().item()
                dormant_units += dormant
                total_units += activity.numel()
    
    # Check value function activations
    if hasattr(vf, 'activations') and vf.activations:
        for key, feat in vf.activations.items():
            if feat is not None and feat.dim() >= 2:
                activity = (feat != 0).float().mean(dim=0)
                dormant = (activity < threshold).sum().item()
                dormant_units += dormant
                total_units += activity.numel()
    
    return dormant_units / total_units if total_units > 0 else 0.0


@torch.no_grad()
def compute_stable_rank_from_features(feature_activity):
    """
    Compute stable rank from feature activations using paper's method.
    
    Args:
        feature_activity: Tensor of shape (num_samples, num_features) - last layer activations
    
    Returns:
        stable_rank: float - approximate rank (99% variance captured)
    """
    if feature_activity is None or feature_activity.numel() == 0:
        return 1.0
    
    # Use the paper's method: compute_matrix_rank_summaries
    _, _, _, stable_rank = compute_matrix_rank_summaries(
        m=feature_activity, 
        prop=0.99, 
        use_scipy=True
    )
    return stable_rank.item() if torch.is_tensor(stable_rank) else float(stable_rank)

In [ ]:
def run_hgsr_gsx_rl(env_name='Ant-v4', seed=42, n_steps=1000000, 
                     h_dim=(256, 256), act_type='ReLU', lr=0.0001,
                     bs=4096, log_interval=1000):  # Increased default bs to 4096
    """
    Run HGSR-GSX with PPO on a Gymnasium environment.
    
    SPEEDUP OPTIONS (for H100):
    - Increase bs (batch size): 4096, 8192, or 16384
    - Reduce n_steps: 10M instead of 50M for quick experiments
    - Note: MuJoCo simulation runs on CPU, not GPU
    
    Args:
        env_name: 'Ant-v4', 'Hopper-v4', 'Walker2d-v4', 'HalfCheetah-v4'
        bs: Batch size (larger = faster per step, more GPU utilization)
    """
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    
    # Create environment
    env = gym.make(env_name)
    o_dim = env.observation_space.shape[0]
    a_dim = env.action_space.shape[0]
    print(f"Environment: {env_name}, Obs: {o_dim}, Act: {a_dim}")
    
    # Create networks
    pol = MLPPolicy(o_dim, a_dim, act_type=act_type, h_dim=list(h_dim), device=device, init='lecun')
    vf = MLPVF(o_dim, act_type=act_type, h_dim=list(h_dim), device=device, init='lecun')
    
    # Create HGSR-GSX learner
    gsx = HGSR_GSX_RL(pol, vf, lr=lr, device=device, bs=bs, h_dim=h_dim)
    
    # Create buffer
    buf = Buffer(o_dim, a_dim, bs, device=device)
    
    # Initialize results tracking
    R = {
        'episodic_returns': [],
        'termination_steps': [],
        'dormant_units': [],
        'weight_magnitude': [],
        'stable_rank': [],
        'hard_resets': [],
        'expansions': [],
        'p_loss': [],
        'v_loss': [],
        'pol_feature_activity': [],
        'vf_feature_activity': [],
        'spectrum_history': [],
        'eigenvalue_history': [],
        'reset_log': [],
        'expand_log': [],
        'hyperparams': {
            'seed': seed, 
            'env_name': env_name, 
            'tau_D': gsx.tau_D, 
            'epsilon': gsx.epsilon,
            'is_continual': n_steps >= 10000000
        }
    }
    
    print(f"\n{'='*60}")
    print(f"HGSR-GSX-RL: {env_name} | Steps: {n_steps:,} | Seed: {seed}")
    print(f"{'='*60}")
    
    # Feature tracking for stable rank (like paper)
    num_layers = len(h_dim)
    short_term_feature_activity = torch.zeros(1000, h_dim[-1], device=device)  # Last layer features
    feature_idx = 0
    
    # Training loop - Gymnasium API
    o, info = env.reset(seed=seed)  # Gymnasium returns (obs, info)
    episode_return = 0
    episode_count = 0
    
    pbar = tqdm(range(n_steps), desc="Training")
    for step in pbar:
        gsx.step = step
        
        # Get action and log features
        with torch.no_grad():
            a, logp, dist = pol.action(
                torch.tensor(o, dtype=torch.float32, device=device).unsqueeze(0),
                to_log_features=True
            )
            
            # Store last HIDDEN layer features for stable rank (not output layer)
            if hasattr(pol, 'activations') and pol.activations:
                features = list(pol.activations.values())
                # Use second-to-last layer (hidden layer), not output
                if len(features) >= 2 and features[-2] is not None:
                    feat = features[-2][0]  # Second to last = last hidden layer
                    if feat.shape[0] == h_dim[-1]:
                        short_term_feature_activity[feature_idx % 1000] = feat
                        feature_idx += 1
                    
        a = a[0].cpu().numpy()
        
        # Step environment - Gymnasium API: returns (obs, reward, terminated, truncated, info)
        op, r, terminated, truncated, info = env.step(a)
        done = terminated or truncated

        episode_return += r
        
        # Store transition
        buf.store(o, a, r, op, logp.cpu().numpy(), pol.dist_to(dist, to_device='cpu'), float(done))
        o = op
        
        # Episode ended
        if done:
            R['episodic_returns'].append(episode_return)
            R['termination_steps'].append(step)
            episode_count += 1
            episode_return = 0
            o, info = env.reset()  # Gymnasium returns (obs, info)
        
        # Learning step when buffer is full
        if len(buf.o_buf) >= bs:
            learn_logs = gsx.learn(buf)
            buf.clear()
            
            R['p_loss'].append(learn_logs['p_loss'])
            R['v_loss'].append(learn_logs['v_loss'])
            
            # Check and intervene
            intervention = gsx.check_and_intervene()
            R['hard_resets'].append(intervention['hard_resets'])
            R['expansions'].append(intervention['expansions'])
            
            # Compute metrics
            R['weight_magnitude'].append(gsx.compute_weight_magnitude())
            R['dormant_units'].append(compute_dormant_units_rl(pol, vf))
        
        # Compute stable rank every 10K steps from short-term feature buffer
        if step > 0 and step % 10000 == 0:
            valid_samples = min(feature_idx, 1000)
            if valid_samples > 10:
                stable_rank = compute_stable_rank_from_features(short_term_feature_activity[:valid_samples])
                R['stable_rank'].append(stable_rank)
        
        # Update progress bar with stats (every 1000 steps)
        if step > 0 and step % 1000 == 0:
            recent_returns = R['episodic_returns'][-10:] if R['episodic_returns'] else [0]
            avg_return = np.mean(recent_returns)
            dormant = R['dormant_units'][-1] * 100 if R['dormant_units'] else 0
            stable_rank_val = R['stable_rank'][-1] if R['stable_rank'] else h_dim[-1]
            pbar.set_postfix({
                'Eps': episode_count,
                'Ret': f'{avg_return:.0f}',
                'Dorm': f'{dormant:.1f}%',
                'SR': f'{stable_rank_val:.0f}',
                'R': gsx.hard_reset_count,
                'E': gsx.expand_count
            })
    
    # Final metrics
    R['total_hard_resets'] = gsx.hard_reset_count
    R['total_expansions'] = gsx.expand_count
    R['spectrum_history'] = gsx.spectrum_history
    R['eigenvalue_history'] = gsx.eigenvalue_history
    R['reset_log'] = gsx.reset_log
    R['expand_log'] = gsx.expand_log
    
    env.close()
    return R

In [ ]:
def run_multi_seed_experiment_rl(env_name='Ant-v3', num_seeds=5, n_steps=1000000,
                                   save_dir='./results_hgsr_gsx_rl'):
    """Run HGSR-GSX-RL experiment with multiple seeds."""
    os.makedirs(save_dir, exist_ok=True)
    
    from mlproj_manager.util import get_random_seeds
    random_seeds = get_random_seeds()
    
    all_results = []
    
    for run_idx in range(num_seeds):
        seed = random_seeds[run_idx]
        print(f"\n{'='*60}")
        print(f"RUN {run_idx+1}/{num_seeds} - Seed: {seed}")
        print(f"{'='*60}")
        
        results = run_hgsr_gsx_rl(env_name=env_name, seed=seed, n_steps=n_steps)
        
        run_path = os.path.join(save_dir, f'run_{run_idx}_seed_{seed}.pkl')
        with open(run_path, 'wb') as f:
            pickle.dump(results, f)
        print(f"✓ Saved: {run_path}")
        
        all_results.append(results)
    
    aggregated = aggregate_rl_results(all_results)
    
    agg_path = os.path.join(save_dir, 'aggregated_results.pkl')
    with open(agg_path, 'wb') as f:
        pickle.dump({'aggregated': aggregated, 'all_results': all_results}, f)
    print(f"\n✓ All runs complete. Saved to {save_dir}")
    
    return aggregated, all_results


def aggregate_rl_results(all_results):
    """Aggregate results from multiple RL runs."""
    num_runs = len(all_results)
    
    aggregated = {
        'num_runs': num_runs,
        'seeds': [r['hyperparams']['seed'] for r in all_results],
        'final_return_mean': np.mean([np.mean(r['episodic_returns'][-100:]) for r in all_results]),
        'final_return_std': np.std([np.mean(r['episodic_returns'][-100:]) for r in all_results]),
        'total_resets_mean': np.mean([r.get('total_hard_resets', 0) for r in all_results]),
        'total_expands_mean': np.mean([r.get('total_expansions', 0) for r in all_results]),
    }
    
    return aggregated

In [ ]:
def plot_rl_results(results, save_prefix='hgsr_gsx_rl', label='HGSR-GSX'):
    """
    Plot results in paper style: 4 horizontal panels.
    1. Total episodic reward
    2. Stable rank (scaled 0-100)
    3. Percentage of dormant units  
    4. Average weight magnitude
    """
    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    
    h_dim_max = results['hyperparams'].get('h_dim', (256,))[-1] if 'hyperparams' in results else 256
    color = 'C0'  # Blue for HGSR-GSX
    
    # 1. Total Episodic Reward
    ax = axes[0]
    returns = results['episodic_returns']
    termination_steps = results.get('termination_steps', list(range(len(returns))))
    
    # Convert to step-based x-axis
    if termination_steps and len(termination_steps) == len(returns):
        x_steps = np.array(termination_steps)
    else:
        x_steps = np.arange(len(returns)) * 1000  # Approximate
    
    ax.plot(x_steps, returns, alpha=0.2, color=color)
    # Rolling average
    if len(returns) > 50:
        window = min(50, len(returns)//4)
        smoothed = np.convolve(returns, np.ones(window)/window, mode='valid')
        x_smooth = x_steps[window//2:window//2+len(smoothed)]
        ax.plot(x_smooth, smoothed, color=color, linewidth=2, label=label)
        # Confidence band (simple std approximation)
        ax.fill_between(x_smooth, smoothed - np.std(returns[:window]), 
                       smoothed + np.std(returns[:window]), alpha=0.3, color=color)
    ax.set_xlabel('Time Step')
    ax.set_ylabel('Return')
    ax.set_title('Total episodic reward')
    ax.grid(True, alpha=0.3)
    ax.legend(loc='upper left')
    
    # 2. Stable Rank (scaled to 0-100)
    ax = axes[1]
    stable_ranks = results.get('stable_rank', [])
    if stable_ranks:
        # Paper scales: stable_rank / h_dim * 100
        scaled_ranks = np.array(stable_ranks) / h_dim_max * 100
        x_sr = np.arange(len(scaled_ranks)) * 10000  # Every 10K steps
        ax.plot(x_sr, scaled_ranks, color=color, linewidth=2, label=label)
        ax.fill_between(x_sr, scaled_ranks*0.95, scaled_ranks*1.05, alpha=0.3, color=color)
    ax.set_xlabel('Time Step')
    ax.set_ylabel('Stable Rank')
    ax.set_title('Stable rank Scaled [0, 100]')
    ax.set_ylim(0, 100)
    ax.grid(True, alpha=0.3)
    
    # 3. Percentage of Dormant Units
    ax = axes[2]
    dormant = np.array(results.get('dormant_units', [])) * 100
    if len(dormant) > 0:
        x_dorm = np.arange(len(dormant))
        ax.plot(x_dorm, dormant, color=color, linewidth=1, alpha=0.5)
        # Smoothed
        if len(dormant) > 20:
            window = min(20, len(dormant)//4)
            smoothed_dorm = np.convolve(dormant, np.ones(window)/window, mode='valid')
            ax.plot(range(window//2, window//2+len(smoothed_dorm)), smoothed_dorm, 
                   color=color, linewidth=2, label=label)
    ax.set_xlabel('Time Step')
    ax.set_ylabel('Dormant (%)')
    ax.set_title('Percentage of dormant units')
    ax.set_ylim(0, max(60, dormant.max()*1.1) if len(dormant) > 0 else 60)
    ax.grid(True, alpha=0.3)
    
    # 4. Average Weight Magnitude
    ax = axes[3]
    weight_mag = results.get('weight_magnitude', [])
    if len(weight_mag) > 0:
        x_wm = np.arange(len(weight_mag))
        ax.plot(x_wm, weight_mag, color=color, linewidth=1, alpha=0.5)
        # Smoothed
        if len(weight_mag) > 20:
            window = min(20, len(weight_mag)//4)
            smoothed_wm = np.convolve(weight_mag, np.ones(window)/window, mode='valid')
            ax.plot(range(window//2, window//2+len(smoothed_wm)), smoothed_wm, 
                   color=color, linewidth=2, label=label)
    ax.set_xlabel('Time Step')
    ax.set_ylabel('Weight Magnitude')
    ax.set_title('Average weight magnitude')
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(f'{save_prefix}_paper_style.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f"✓ Saved: {save_prefix}_paper_style.png")


def plot_rl_results_detailed(results, save_prefix='hgsr_gsx_rl'):
    """Plot detailed results including losses and interventions."""
    fig, axes = plt.subplots(2, 3, figsize=(16, 10))
    
    # 1. Episodic Returns
    ax = axes[0, 0]
    returns = results['episodic_returns']
    ax.plot(returns, alpha=0.3, color='blue')
    if len(returns) > 100:
        smoothed = np.convolve(returns, np.ones(100)/100, mode='valid')
        ax.plot(range(50, 50+len(smoothed)), smoothed, color='blue', linewidth=2)
    ax.set_xlabel('Episode')
    ax.set_ylabel('Return')
    ax.set_title(f"Episodic Return (Final: {np.mean(returns[-100:]) if returns else 0:.1f})")
    ax.grid(True, alpha=0.3)
    
    # 2. Dormant Units
    ax = axes[0, 1]
    dormant = np.array(results.get('dormant_units', [])) * 100
    if len(dormant) > 0:
        ax.plot(dormant, 'r-', linewidth=1)
        ax.set_title(f"Dormant Neurons (Final: {dormant[-1]:.1f}%)")
    ax.set_xlabel('Update')
    ax.set_ylabel('Dormant (%)')
    ax.grid(True, alpha=0.3)
    
    # 3. Weight Magnitude
    ax = axes[0, 2]
    weight_mag = results.get('weight_magnitude', [])
    if weight_mag:
        ax.plot(weight_mag, 'g-', linewidth=1)
    ax.set_xlabel('Update')
    ax.set_ylabel('Weight Magnitude')
    ax.set_title('Average Weight Magnitude')
    ax.grid(True, alpha=0.3)
    
    # 4. Cumulative Resets & Expansions
    ax = axes[1, 0]
    hard_resets = results.get('hard_resets', [])
    expansions = results.get('expansions', [])
    if hard_resets:
        cum_resets = np.cumsum(hard_resets)
        cum_expands = np.cumsum(expansions)
        ax.plot(cum_resets, 'orange', linewidth=2, label='Resets')
        ax.plot(cum_expands, 'purple', linewidth=2, label='Expands')
        ax.legend()
    ax.set_xlabel('Update')
    ax.set_ylabel('Cumulative Count')
    ax.set_title(f"Interventions (R:{results.get('total_hard_resets', 0)}, E:{results.get('total_expansions', 0)})")
    ax.grid(True, alpha=0.3)
    
    # 5. Stable Rank
    ax = axes[1, 1]
    stable_ranks = results.get('stable_rank', [])
    if stable_ranks:
        ax.plot(stable_ranks, 'b-o', linewidth=2)
    ax.set_xlabel('Checkpoint (every 10K steps)')
    ax.set_ylabel('Stable Rank')
    ax.set_title('Stable Rank Evolution')
    ax.grid(True, alpha=0.3)
    
    # 6. Loss
    ax = axes[1, 2]
    p_loss = results.get('p_loss', [])
    v_loss = results.get('v_loss', [])
    if p_loss:
        ax.plot(p_loss, 'c-', alpha=0.5, label='Policy Loss')
        ax.plot(v_loss, 'm-', alpha=0.5, label='Value Loss')
        ax.legend()
    ax.set_xlabel('Update')
    ax.set_ylabel('Loss')
    ax.set_title('Training Losses')
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(f'{save_prefix}_detailed.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f"✓ Saved: {save_prefix}_detailed.png")


def plot_spectrum_tracking(results, save_prefix='hgsr_gsx_rl'):
    """Plot spectrum (D, S ratio) tracking."""
    import pandas as pd
    
    if not results.get('spectrum_history'):
        print("No spectrum history to plot")
        return
        
    df = pd.DataFrame(results['spectrum_history'])
    if len(df) == 0:
        return
    
    layers = df['layer'].unique()[:4]
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle('Spectrum Tracking (D_ratio, S_ratio)', fontsize=14, fontweight='bold')
    
    for i, layer in enumerate(layers):
        ax = axes[i//2, i%2]
        ld = df[df['layer'] == layer]
        ax.plot(ld['step'], ld['D_ratio'], 'o-', label='D_ratio', ms=2, alpha=0.8)
        ax.plot(ld['step'], ld['S_ratio'], 's-', label='S_ratio', ms=2, alpha=0.8)
        ax.axhline(0.4, c='r', ls='--', alpha=0.5, label='τ_D=0.4')
        ax.axhline(0.15, c='g', ls='--', alpha=0.5, label='ε=0.15')
        short_name = layer.split('.')[-1] if '.' in layer else layer[:25]
        ax.set_title(short_name)
        ax.legend(loc='upper right', fontsize=8)
        ax.set_ylim(0, 1.5)
        ax.set_xlabel('Step')
        ax.set_ylabel('Ratio')
        ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(f'{save_prefix}_spectrum.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f"✓ Saved: {save_prefix}_spectrum.png")

## Run Experiment

### Configuration (modify these for your experiment)

In [ ]:
# ============================================================
# CONFIGURATION - Modify these settings for your experiment
# ============================================================
CONFIG = {
    'env_name': 'Ant-v4',          # 'Ant-v4', 'Hopper-v4', 'Walker2d-v4', 'HalfCheetah-v4'
    'n_steps': 50000000,             # Quick test (paper uses 100M steps)
    'seed': 42,
    'save_dir': './results_hgsr_gsx_rl',
    'num_seeds': 1,
}

In [ ]:
def run_experiment(config=None):
    """Run experiment with given config"""
    if config is None:
        config = CONFIG
    
    if config.get('num_seeds', 1) > 1:
        aggregated, all_results = run_multi_seed_experiment_rl(
            env_name=config['env_name'],
            num_seeds=config['num_seeds'],
            n_steps=config['n_steps'],
            save_dir=config['save_dir']
        )
        plot_rl_results(all_results[0], save_prefix=os.path.join(config['save_dir'], 'hgsr_gsx_rl'))
        return aggregated, all_results
    else:
        results = run_hgsr_gsx_rl(
            env_name=config['env_name'],
            seed=config['seed'],
            n_steps=config['n_steps']
        )
        
        # Save results
        os.makedirs(config['save_dir'], exist_ok=True)
        save_path = os.path.join(config['save_dir'], f"results_seed_{config['seed']}.pkl")
        with open(save_path, 'wb') as f:
            pickle.dump(results, f)
        print(f"✓ Saved results to: {save_path}")
        
        # Plot
        plot_rl_results(results, save_prefix=os.path.join(config['save_dir'], 'hgsr_gsx_rl'))
        plot_spectrum_tracking(results, save_prefix=os.path.join(config['save_dir'], 'hgsr_gsx_rl'))
        
        # Summary
        print(f"\n{'='*60}")
        print("SUMMARY")
        print(f"{'='*60}")
        print(f"Episodes: {len(results['episodic_returns'])}")
        if results['episodic_returns']:
            print(f"Final Return (last 100): {np.mean(results['episodic_returns'][-100:]):.1f}")
        if results['dormant_units']:
            print(f"Final Dormant: {results['dormant_units'][-1]*100:.1f}%")
        print(f"Total Resets: {results['total_hard_resets']}")
        print(f"Total Expansions: {results['total_expansions']}")
        print(f"{'='*60}")
        
        return results

In [ ]:
# Run experiment
results = run_experiment()